<a href="https://colab.research.google.com/github/tacochan0129/helpyouchoose/blob/main/ihg_reward_radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json

# 1. 你找到的隱藏 API 網址
url = "https://apis.ihg.com/availability/v1/calendar"

# 2. 準備你要發送的查詢條件 (Payload)
payload = {
    "hotelMnemonics": ["OSAKB"],
    "startDate": "2026-03-18",
    "endDate": "2026-05-18",
    "lengthOfStay": 1,
    "guestCounts": [
        {
            "otaCode": "AQC10",
            "count": 1
        }
    ],
    "options": {
        "includeSellStrategy": "followChannel",
        "returnAmountsAfterTaxForLowestOffer": True,
        "returnAverages": True
    },
    "rates": {
        "ratePlanCodes": ["IVAN1", "IVAN3", "IVAN5", "IVAN6", "IVAN7", "IVANI"]
    }
}

# 3. 準備 Headers (很重要！讓伺服器以為我們是正常的 Chrome 瀏覽器)
headers = {
    'sec-ch-ua-platform': '"Windows"',
    'Referer': 'https://www.ihg.com/',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'ihg-language': 'en-US',
    'sec-ch-ua-mobile': '?0',
    'x-ihg-api-key': 'se9ym5iAzaW8pxfBjkmgbuGjJcr3Pj6Y',
    'IHG-SessionId': '587984dc-9d3d-435c-85f0-d6c2366c9870',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'content-type': 'application/json; charset=UTF-8',
    'IHG-TransactionId': 'd8a0afb2-0909-4949-80aa-a233266bcf1e',
}

json_data = {
    'hotelMnemonics': [
        'OSAKB',
    ],
    'startDate': '2026-03-18',
    'endDate': '2026-05-18',
    'lengthOfStay': 1,
    'guestCounts': [
        {
            'otaCode': 'AQC10',
            'count': 1,
        },
    ],
    'options': {
        'includeSellStrategy': 'followChannel',
        'returnAmountsAfterTaxForLowestOffer': True,
        'returnAverages': True,
        'lowestOfferPerRatePlan': True,
        'identifyLowestOfferPerRatePlan': True,
    },
    'rates': {
        'ratePlanCodes': [
            'IVAN1',
            'IVAN3',
            'IVAN5',
            'IVAN6',
            'IVAN7',
            'IVANI',
        ],
    },
}

response = requests.post('https://apis.ihg.com/availability/v1/calendar', headers=headers, json=json_data)

# Note: json_data will not be serialized by requests
# exactly as it was in the original request.
#data = '{"hotelMnemonics":["OSAKB"],"startDate":"2026-03-18","endDate":"2026-05-18","lengthOfStay":1,"guestCounts":[{"otaCode":"AQC10","count":1}],"options":{"includeSellStrategy":"followChannel","returnAmountsAfterTaxForLowestOffer":true,"returnAverages":true,"lowestOfferPerRatePlan":true,"identifyLowestOfferPerRatePlan":true},"rates":{"ratePlanCodes":["IVAN1","IVAN3","IVAN5","IVAN6","IVAN7","IVANI"]}}'
#response = requests.post('https://apis.ihg.com/availability/v1/calendar', headers=headers, data=data)



# 4. 使用 POST 方法發送請求
print("正在向 IHG 伺服器發送請求...")
response = requests.post(url, json=payload, headers=headers)

# 5. 檢查結果
if response.status_code == 200:
    print("✅ 成功抓取資料！")
    data = response.json() # 將回傳的文字轉換為 Python 的字典格式

    # 這裡我們先印出前 300 個字元確認格式，避免資料太長洗版
    print(json.dumps(data, indent=2, ensure_ascii=False)[:300] + "...\n")
else:
    print(f"❌ 抓取失敗，伺服器回傳狀態碼：{response.status_code}")
    print("錯誤訊息：", response.text)

正在向 IHG 伺服器發送請求...
✅ 成功抓取資料！
{
  "data": {
    "startDate": "2026-03-18",
    "endDate": "2026-05-18",
    "lengthOfStay": 1,
    "guestCounts": [
      {
        "guestType": "ADULT",
        "otaCode": "AQC10",
        "count": 1
      }
    ],
    "hotels": [
      {
        "hotel": {
          "brandCode": "CP",
          ...



In [ ]:
import json

# 1. 將完整的資料存成一個實體檔案，方便我們隨時查看完整結構
with open("ihg_full_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print("📁 完整的 JSON 資料已經存成 'ihg_full_data.json'！")
print("👉 (你可以點擊 Colab 左側選單的「資料夾」圖示，點開該檔案來看看原本到底有多長)\n")

# 2. 用 Python 一層一層往深處挖，把報價找出來
hotels_info = data.get("data", {}).get("hotels", [])

if hotels_info:
    # 進入第一間飯店 (神戶 ANA 皇冠假日酒店) 的資料區塊
    first_hotel = hotels_info[0]

    # 報價通常藏在 ratePlans (費率方案) 這個陣列裡面
    rate_plans = first_hotel.get("ratePlans", [])

    print(f"🏨 總共找到 {len(rate_plans)} 種費率方案！\n")

    # 針對每一個方案，我們印出前 3 天的報價來確認格式
    for plan in rate_plans:
        plan_code = plan.get("ratePlanCode", "未知方案")
        print(f"🏷️ 方案代碼: {plan_code}")

        # 方案裡面通常會有一個 calendar 陣列，記錄這兩個月每一天的狀況
        daily_rates = plan.get("calendar", [])

        for day in daily_rates[:3]: # 只拿前 3 天當範例
            date = day.get("date")
            status = day.get("status") # 通常會顯示 AVAILABLE (有房) 或 SOLD_OUT (滿房)

            # 尋找金額或點數 (不同飯店或不同方案的欄位名稱可能略有不同)
            price_info = day.get("price", {})
            amount = price_info.get("amount", "無")
            currency = price_info.get("currency", "")
            points = day.get("points", "") # 如果是點數房，可能會有這個欄位

            if points:
                print(f"  - 日期 {date}: [{status}], 需要點數: {points}")
            elif amount != "無":
                print(f"  - 日期 {date}: [{status}], 現金價: {amount} {currency}")
            else:
                print(f"  - 日期 {date}: [{status}], (此日期無提供價格)")
        print("-" * 30)
else:
    print("找不到 hotels 欄位，請檢查 JSON 結構。")

📁 完整的 JSON 資料已經存成 'ihg_full_data.json'！
👉 (你可以點擊 Colab 左側選單的「資料夾」圖示，點開該檔案來看看原本到底有多長)

🏨 總共找到 18 種費率方案！

🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------
🏷️ 方案代碼: 未知方案
------------------------------


In [ ]:
import json

# 1. 讀取我們剛剛存下來的完整資料檔案
with open("ihg_full_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

hotels_info = data.get("data", {}).get("hotels", [])

if hotels_info:
    rate_plans = hotels_info[0].get("ratePlans", [])

    if rate_plans:
        print("🔍 抓到了！讓我們來看看第一個方案的真實結構：\n")
        # 把第一個方案 (索引值為 0) 轉成漂亮的 JSON 格式印出來
        # 為了避免太長，我們印出前 1500 個字元就好
        print(json.dumps(rate_plans[0], indent=2, ensure_ascii=False)[:1500])
    else:
        print("找不到任何費率方案。")

🔍 抓到了！讓我們來看看第一個方案的真實結構：

{
  "code": "IVANI",
  "isFreeNight": false,
  "isRewardNight": true,
  "areAmtsConfidential": true
}


In [ ]:
import json

with open("ihg_full_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

first_hotel = data.get("data", {}).get("hotels", [])[0]

print("🔍 這家飯店的資料夾裡面，總共有這些主要欄位 (Keys)：")
for key in first_hotel.keys():
    print(f"- {key}")

🔍 這家飯店的資料夾裡面，總共有這些主要欄位 (Keys)：
- hotel
- ratePlans
- calendar


In [ ]:
import json

# 讀取完整資料
with open("ihg_full_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 直接鎖定第一家飯店的 calendar 資料夾
calendar_data = data.get("data", {}).get("hotels", [])[0].get("calendar", [])

if calendar_data:
    print("📅 成功打開日曆！讓我們看看「第一天」的報價結構長什麼樣子：\n")
    # 印出第一天 (索引值為 0) 的完整資料
    print(json.dumps(calendar_data[0], indent=2, ensure_ascii=False))
else:
    print("日曆裡面沒有資料耶！")

📅 成功打開日曆！讓我們看看「第一天」的報價結構長什麼樣子：

{
  "start": "2026-03-18",
  "end": "2026-03-18",
  "offers": [
    {
      "id": 1,
      "ratePlanCode": "IVANI",
      "inventoryTypesAvailable": [
        {
          "inventoryTypeCode": "CDXG",
          "numberOfAvailableProducts": 5
        }
      ],
      "checkInPoints": 42000.0,
      "averageDailyPoints": 42000.0,
      "totalPoints": 42000.0
    },
    {
      "id": 2,
      "ratePlanCode": "IVANI",
      "inventoryTypesAvailable": [
        {
          "inventoryTypeCode": "CSTG",
          "numberOfAvailableProducts": 9
        }
      ],
      "checkInPoints": 26000.0,
      "averageDailyPoints": 26000.0,
      "totalPoints": 26000.0
    },
    {
      "id": 3,
      "ratePlanCode": "IVANI",
      "inventoryTypesAvailable": [
        {
          "inventoryTypeCode": "OEXN",
          "numberOfAvailableProducts": 1
        }
      ],
      "checkInPoints": 56000.0,
      "averageDailyPoints": 56000.0,
      "totalPoints": 56000.0
    },
 

In [ ]:
import requests
import json

# 1. API 網址與查詢條件 (這裡以你查詢的 3/18 到 5/18 為例)
url = "https://apis.ihg.com/availability/v1/calendar"
payload = {
    "hotelMnemonics": ["OSAKB"], # 神戶 ANA 皇冠假日酒店
    "startDate": "2026-05-18",
    "endDate": "2026-05-22",
    "lengthOfStay": 1,
    "guestCounts": [{"otaCode": "AQC10", "count": 1}],
    "options": {
        "includeSellStrategy": "followChannel",
        "returnAmountsAfterTaxForLowestOffer": True,
        "returnAverages": True
    },
    "rates": {
        "ratePlanCodes": ["IVANI"] # 我們直接在這裡鎖定只查 IVANI 方案！
    }
}

import requests

headers = {
    'sec-ch-ua-platform': '"Windows"',
    'Referer': 'https://www.ihg.com/',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'ihg-language': 'en-US',
    'sec-ch-ua-mobile': '?0',
    'x-ihg-api-key': 'se9ym5iAzaW8pxfBjkmgbuGjJcr3Pj6Y',
    'IHG-SessionId': '587984dc-9d3d-435c-85f0-d6c2366c9870',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'content-type': 'application/json; charset=UTF-8',
    'IHG-TransactionId': 'be26e558-b489-4b66-9fa8-3dd1f6bc3450',
}

json_data = {
    'hotelMnemonics': [
        'OSAKB',
    ],
    'startDate': '2026-03-18',
    'endDate': '2026-05-18',
    'lengthOfStay': 1,
    'guestCounts': [
        {
            'otaCode': 'AQC10',
            'count': 1,
        },
    ],
    'options': {
        'includeSellStrategy': 'followChannel',
        'returnAmountsAfterTaxForLowestOffer': True,
        'returnAverages': True,
        'lowestOfferPerRatePlan': True,
        'identifyLowestOfferPerRatePlan': True,
    },
    'rates': {
        'ratePlanCodes': [
            'IVAN1',
            'IVAN3',
            'IVAN5',
            'IVAN6',
            'IVAN7',
            'IVANI',
        ],
    },
}

response = requests.post('https://apis.ihg.com/availability/v1/calendar', headers=headers, json=json_data)

# Note: json_data will not be serialized by requests
# exactly as it was in the original request.
#data = '{"hotelMnemonics":["OSAKB"],"startDate":"2026-03-18","endDate":"2026-05-18","lengthOfStay":1,"guestCounts":[{"otaCode":"AQC10","count":1}],"options":{"includeSellStrategy":"followChannel","returnAmountsAfterTaxForLowestOffer":true,"returnAverages":true,"lowestOfferPerRatePlan":true,"identifyLowestOfferPerRatePlan":true},"rates":{"ratePlanCodes":["IVAN1","IVAN3","IVAN5","IVAN6","IVAN7","IVANI"]}}'
#response = requests.post('https://apis.ihg.com/availability/v1/calendar', headers=headers, data=data)


print("正在向 IHG 伺服器發送請求...")
response = requests.post(url, json=payload, headers=headers)

if response.status_code == 200:
    print("✅ 成功抓取資料！開始解析點數房...\n")
    data = response.json()

    # 解析我們剛剛摸透的結構
    calendar_data = data.get("data", {}).get("hotels", [])[0].get("calendar", [])

    # 建立一個清單來儲存找到的點數房資訊
    found_reward_nights = []

    for day in calendar_data:
        date = day.get("start")
        offers = day.get("offers", [])

        day_results = []
        for offer in offers:
            # 再次確認是 IVANI 方案
            if offer.get("ratePlanCode") == "IVANI":
                # 抓取房型代碼 (可能有多個，我們取第一個)
                room_types = offer.get("inventoryTypesAvailable", [])
                room_code = room_types[0].get("inventoryTypeCode") if room_types else "未知房型"

                # 抓取所需點數
                points = offer.get("totalPoints")

                if points:
                    day_results.append(f"房型 {room_code}: {int(points)} 點")

        # 如果這一天有找到任何點數房，就記錄下來
        if day_results:
            result_string = f"📅 {date} | " + " / ".join(day_results)
            found_reward_nights.append(result_string)
            print(result_string)

    if not found_reward_nights:
         print("❌ 在這段期間內沒有找到任何 IVANI 點數房釋出。")

else:
    print(f"❌ 抓取失敗，伺服器回傳狀態碼：{response.status_code}")

正在向 IHG 伺服器發送請求...
✅ 成功抓取資料！開始解析點數房...

📅 2026-05-18 | 房型 CDXG: 44000 點 / 房型 CSTG: 28000 點 / 房型 KNGN: 28000 點 / 房型 KNGS: 30000 點 / 房型 OAFS: 21000 點 / 房型 OAQN: 17000 點 / 房型 OEEN: 50000 點 / 房型 OEXN: 58000 點 / 房型 OSIN: 17000 點 / 房型 OSIS: 17000 點 / 房型 OSNN: 21000 點 / 房型 OSNS: 23000 點 / 房型 TAES: 30000 點 / 房型 TANN: 28000 點 / 房型 TAYS: 42000 點 / 房型 TAZN: 40000 點 / 房型 TBIS: 46000 點 / 房型 TBJN: 44000 點 / 房型 TDBN: 34000 點 / 房型 TDBS: 36000 點 / 房型 TEEN: 58000 點 / 房型 TLON: 48000 點 / 房型 TLOS: 53000 點 / 房型 TTWN: 21000 點 / 房型 TTWS: 23000 點 / 房型 XAKN: 138000 點 / 房型 XBGN: 158000 點 / 房型 XOTN: 70000 點
📅 2026-05-19 | 房型 CDXG: 44000 點 / 房型 CSTG: 28000 點 / 房型 KNGN: 28000 點 / 房型 KNGS: 30000 點 / 房型 OAFS: 21000 點 / 房型 OAQN: 17000 點 / 房型 OEEN: 50000 點 / 房型 OEXN: 58000 點 / 房型 OSIN: 17000 點 / 房型 OSIS: 17000 點 / 房型 OSNN: 21000 點 / 房型 OSNS: 23000 點 / 房型 TAES: 30000 點 / 房型 TANN: 28000 點 / 房型 TAYS: 42000 點 / 房型 TAZN: 40000 點 / 房型 TBIS: 46000 點 / 房型 TBJN: 44000 點 / 房型 TDBN: 34000 點 / 房型 TDBS: 36000 點 / 房型 TEEN: 58000 點 /